# Vector stores for RAG — companion notebook

**Course:** AI RAG Agents — Meeting 10-11

End-to-end walkthrough on the `jamescalam/ai-arxiv-chunked` dataset:

1. **Embed** 5,000 arxiv chunks with an open-source model (cached to disk)
2. **Chroma** — store with metadata, query, filter by category and year
3. **Qdrant** (`:memory:`) — the same task, with payload filters
4. **Tune HNSW** — sweep `ef_search`, plot recall vs latency
5. **Metadata exercise** — add a new field, filter on it, compare backends

---

### Setup

In [ ]:
# Run once if needed:
# %pip install -q datasets sentence-transformers chromadb qdrant-client matplotlib tqdm

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

DATA_DIR = Path("./data_cache")
DATA_DIR.mkdir(exist_ok=True)

N_CHUNKS = 5_000          # how much of the dataset to use
EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # 384-dim, MIT-licensed, fast on CPU
EMBED_DIM = 384

## 1. Load the dataset

`jamescalam/ai-arxiv-chunked` is ~41k chunks of AI/ML arxiv papers, with metadata (title, authors, categories, dates). We take the first 5,000 chunks for the demo.

In [ ]:
from datasets import load_dataset

ds = load_dataset("jamescalam/ai-arxiv-chunked", split="train")
print("Total rows:", len(ds))
print("Columns   :", ds.column_names)
print("\nSample row:")
row = ds[0]
for k, v in row.items():
    s = str(v)
    print(f"  {k:<18s} {s[:90] + ('…' if len(s) > 90 else '')}")

In [ ]:
# Take a stable subset for reproducibility
rows = ds.select(range(N_CHUNKS))

# We'll keep this exact view through the rest of the notebook
texts: list[str] = [r["chunk"] for r in rows]

# Build the metadata records we'll store with each vector.
# Rule: keep only what we'll filter on or display. No mass copy of every column.
def make_metadata(r: dict) -> dict:
    pub = (r.get("published") or "")[:10]   # ISO date string -> YYYY-MM-DD
    year = int(pub[:4]) if pub[:4].isdigit() else 0
    return {
        "doc_id":   r.get("id") or r.get("doi") or "",
        "title":    (r.get("title") or "").strip(),
        "category": r.get("primary_category") or "",
        "year":     year,
        "source":   r.get("source") or "",
    }

metadatas: list[dict] = [make_metadata(r) for r in rows]
ids: list[str] = [f"chunk_{i:05d}" for i in range(N_CHUNKS)]

print(f"Prepared {len(texts):,} chunks")
print("Example metadata:", metadatas[0])

## 2. Embed (open-source, cached)

We use `BAAI/bge-small-en-v1.5` — 384-dim, runs on CPU in a few minutes for 5k chunks. We cache the array to a `.npy` file so re-running this notebook is instant.

In [ ]:
from sentence_transformers import SentenceTransformer

embeddings_path = DATA_DIR / f"embeddings_{N_CHUNKS}_{EMBED_MODEL.replace('/', '_')}.npy"

if embeddings_path.exists():
    embeddings = np.load(embeddings_path)
    print(f"Loaded cached embeddings: {embeddings.shape}")
else:
    print("Embedding from scratch — first run only…")
    model = SentenceTransformer(EMBED_MODEL)
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,    # cosine similarity becomes dot product
        convert_to_numpy=True,
    )
    np.save(embeddings_path, embeddings)
    print(f"Saved {embeddings.shape} -> {embeddings_path}")

assert embeddings.shape == (N_CHUNKS, EMBED_DIM)

In [ ]:
# A small helper to embed query strings on demand
_query_model = None

def embed_query(text: str) -> np.ndarray:
    global _query_model
    if _query_model is None:
        _query_model = SentenceTransformer(EMBED_MODEL)
    # bge models recommend a query prefix for retrieval
    prefixed = f"Represent this sentence for searching relevant passages: {text}"
    return _query_model.encode(prefixed, normalize_embeddings=True).astype(np.float32)

---
## 3. ChromaDB

Three calls cover most of what we'll do: `add`, `query`, `update/delete`. Persistence is a folder on disk.

In [ ]:
import chromadb

chroma_path = DATA_DIR / "chroma_db"
chroma = chromadb.PersistentClient(path=str(chroma_path))

# Reset for re-runs so the demo is reproducible
if "arxiv" in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection("arxiv")

col = chroma.create_collection(
    name="arxiv",
    metadata={"hnsw:space": "cosine"},
)
print("Collection:", col.name)

In [ ]:
# Insert in batches — Chroma's add() is sync; batching keeps memory steady
BATCH = 500
for i in tqdm(range(0, N_CHUNKS, BATCH), desc="Chroma add"):
    j = min(i + BATCH, N_CHUNKS)
    col.add(
        ids=ids[i:j],
        documents=texts[i:j],
        embeddings=embeddings[i:j].tolist(),
        metadatas=metadatas[i:j],
    )

print("Total in collection:", col.count())

In [ ]:
# Query 1: plain similarity search
q = "What is self-attention and why does it work?"
qv = embed_query(q)

res = col.query(query_embeddings=[qv.tolist()], n_results=5)

for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"[{dist:.3f}] {meta['title'][:60]}  ({meta['category']}, {meta['year']})")
    print("     ", doc[:140].replace("\n", " "), "…")
    print()

In [ ]:
# Query 2: filter by metadata. Chroma uses Mongo-style operators.
res = col.query(
    query_embeddings=[qv.tolist()],
    n_results=5,
    where={
        "$and": [
            {"category": {"$eq": "cs.CL"}},
            {"year":     {"$gte": 2020}},
        ]
    },
)

print("Filtered to category=cs.CL AND year>=2020:\n")
for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
    print(f"  [{dist:.3f}] {meta['title'][:60]}  ({meta['category']}, {meta['year']})")

In [ ]:
# Update / delete by id — useful when re-ingesting changed docs
col.update(ids=["chunk_00000"], metadatas=[{**metadatas[0], "reviewed": True}])
got = col.get(ids=["chunk_00000"])
print("Updated metadata:", got["metadatas"][0])

---
## 4. Qdrant (`:memory:`)

Same data, same task. Notice:
- Vectors and **payloads** are stored together as `PointStruct`s
- Filters are first-class via `Filter` / `FieldCondition` / `Range` / `MatchValue`
- The client is identical for in-memory, local server, or cloud — only the URL changes

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, Range, MatchValue,
    HnswConfigDiff, SearchParams,
)

qdrant = QdrantClient(":memory:")
COLL = "arxiv"

qdrant.recreate_collection(
    collection_name=COLL,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=16, ef_construct=64),  # defaults made explicit
)
print("Created collection:", COLL)

In [ ]:
# Upsert in batches as PointStructs (id, vector, payload)
BATCH = 500
for i in tqdm(range(0, N_CHUNKS, BATCH), desc="Qdrant upsert"):
    j = min(i + BATCH, N_CHUNKS)
    points = [
        PointStruct(
            id=k,                          # qdrant ids: int or UUID
            vector=embeddings[k].tolist(),
            payload={**metadatas[k], "text": texts[k]},  # text lives in payload
        )
        for k in range(i, j)
    ]
    qdrant.upsert(collection_name=COLL, points=points)

info = qdrant.get_collection(COLL)
print("Points in collection:", info.points_count)

In [ ]:
# Plain similarity search
hits = qdrant.query_points(
    collection_name=COLL,
    query=qv.tolist(),
    limit=5,
    with_payload=True,
).points

for h in hits:
    p = h.payload
    print(f"[{h.score:.3f}] {p['title'][:60]}  ({p['category']}, {p['year']})")

In [ ]:
# Filter by payload — same intent as the Chroma query above
hits = qdrant.query_points(
    collection_name=COLL,
    query=qv.tolist(),
    limit=5,
    with_payload=True,
    query_filter=Filter(
        must=[
            FieldCondition(key="category", match=MatchValue(value="cs.CL")),
            FieldCondition(key="year",     range=Range(gte=2020)),
        ]
    ),
).points

print("Filtered to category=cs.CL AND year>=2020:\n")
for h in hits:
    p = h.payload
    print(f"  [{h.score:.3f}] {p['title'][:60]}  ({p['category']}, {p['year']})")

In [ ]:
# Payload indexes make filters fast at scale.
# Without them, selective filters degrade to a payload scan.
qdrant.create_payload_index(collection_name=COLL, field_name="category", field_schema="keyword")
qdrant.create_payload_index(collection_name=COLL, field_name="year",     field_schema="integer")
print("Payload indexes created on 'category' and 'year'.")

---
## 5. ANN tuning — recall vs. latency

We sweep Qdrant's per-query `hnsw_ef` parameter. For each value we measure:

- **Recall@10** against an exact (brute-force) ground truth
- **Median latency**

This is the workflow you'll use any time you tune an ANN index in production.

In [ ]:
# Build a tiny eval set: 50 queries from the dataset itself.
# Using existing chunks as queries is a quick proxy — for real eval, write or sample real user queries.
rng = np.random.default_rng(42)
eval_idx = rng.choice(N_CHUNKS, size=50, replace=False)
eval_vectors = embeddings[eval_idx]

# Exact top-K via brute-force matrix multiply (vectors are normalized -> cosine = dot)
K = 10
sims = eval_vectors @ embeddings.T
exact_topk = np.argpartition(-sims, K, axis=1)[:, :K]
# sort each row by actual similarity
for i, row in enumerate(exact_topk):
    order = np.argsort(-sims[i, row])
    exact_topk[i] = row[order]

exact_sets = [set(r.tolist()) for r in exact_topk]
print(f"Built ground truth for {len(eval_sets := exact_sets)} queries (top-{K}).")

In [ ]:
def measure(ef: int) -> tuple[float, float]:
    """Return (recall@10, median latency in ms) for a given hnsw_ef."""
    recalls, lats = [], []
    for vec, truth in zip(eval_vectors, eval_sets):
        t0 = time.perf_counter()
        hits = qdrant.query_points(
            collection_name=COLL,
            query=vec.tolist(),
            limit=K,
            search_params=SearchParams(hnsw_ef=ef),
            with_payload=False,
        ).points
        lats.append((time.perf_counter() - t0) * 1000)
        approx = {h.id for h in hits}
        recalls.append(len(approx & truth) / K)
    return float(np.mean(recalls)), float(np.median(lats))

ef_values = [10, 20, 40, 80, 160, 320]
results = []
for ef in ef_values:
    r, l = measure(ef)
    print(f"hnsw_ef={ef:>4d}   recall@10={r:.3f}   median_latency={l:.2f} ms")
    results.append((ef, r, l))

In [ ]:
import matplotlib.pyplot as plt

efs   = [r[0] for r in results]
rec   = [r[1] for r in results]
lat   = [r[2] for r in results]

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(efs, rec, "o-", color="#2d5a8e", label="recall@10")
ax1.set_xlabel("hnsw_ef (search effort)")
ax1.set_ylabel("recall@10", color="#2d5a8e")
ax1.set_xscale("log")
ax1.set_ylim(0, 1.02)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(efs, lat, "s--", color="#F5A623", label="latency (ms)")
ax2.set_ylabel("median latency (ms)", color="#F5A623")

plt.title("HNSW: recall–latency tradeoff")
fig.tight_layout()
plt.show()

**Reading the curve:** recall climbs sharply at small `ef`, then plateaus near 1.0. Latency rises roughly linearly. The **elbow** is your operating point: the smallest `ef` where recall meets your target.

On 5,000 chunks the dataset is small and the curve flattens early. On a 1M+ collection, the same shape holds but the elbow shifts right (you need more search effort).

> **Lesson:** tune `ef_search` per query, not per dataset. If the same backend serves both "quick autocomplete" and "deep research" queries, use different `ef` values per call.

---
## 6. Metadata patterns — pre-filter vs. post-filter

When a filter is **selective** (matches few rows), pre-filtering wins. When it's **loose**, post-filtering is fine. Qdrant decides for you; let's just see the difference in practice.

In [ ]:
from collections import Counter

cat_counts = Counter(m["category"] for m in metadatas)
print("Top categories in our 5k subset:")
for c, n in cat_counts.most_common(8):
    print(f"  {c:<10s} {n}")

# Pick one rare and one common category
common = cat_counts.most_common(1)[0][0]
rare   = cat_counts.most_common()[-1][0]
print(f"\nCommon: {common}    Rare: {rare}")

In [ ]:
def time_filtered_query(category: str, n_runs: int = 20) -> float:
    flt = Filter(must=[FieldCondition(key="category", match=MatchValue(value=category))])
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        qdrant.query_points(
            collection_name=COLL, query=qv.tolist(), limit=10,
            query_filter=flt, with_payload=False,
        )
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

print(f"common ({common:<10s}, {cat_counts[common]:>4d} chunks): {time_filtered_query(common):.2f} ms")
print(f"rare   ({rare:<10s}, {cat_counts[rare]:>4d} chunks): {time_filtered_query(rare):.2f} ms")

On this dataset both are sub-millisecond, but the principle is real:

- **Loose filter** → post-filter wins (most ANN candidates pass anyway)
- **Selective filter** → pre-filter on a payload index wins (don't waste time on candidates that'll be dropped)

Qdrant's planner picks adaptively when you have a payload index. Without one, selective filters degrade to a full scan.

---
## 7. Your turn — exercise

1. **Add a derived metadata field** — e.g., `decade = (year // 10) * 10`. Re-ingest a small slice with this added field.
2. **Filter on it** — "papers from the 2010s about transformers" — in **both** Chroma and Qdrant. Compare the syntax.
3. **Measure recall@10** against an exact baseline for `hnsw_ef ∈ {8, 16}` (deliberately too small). Is the recall drop noticeable?
4. **Pick an `hnsw_ef` budget** for a hypothetical p95 latency target of 5 ms. Justify with the curve from §5.

**Stretch:** turn the chunk text into a *sparse* representation (BM25 or splade) and try Qdrant hybrid search — query both dense and sparse vectors and fuse with RRF.

---
## Recap

- A vector store is **persistence + ANN + metadata filters**, not just `argsort`
- **Chroma** for prototypes (3 calls), **Qdrant** for production (same shape, more knobs)
- **HNSW** is the modern default. Tune **`ef_search`** per query; trust the build defaults
- **Measure recall on a sample** before you tune anything
- **Metadata is a schema** — design it on purpose; index fields you filter on